In [1]:
import pandas as pd

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
print(df.shape)
df.head()

(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [3]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print(df["TotalCharges"].isnull().sum())
df[df["TotalCharges"].isnull()][["customerID", "tenure", "MonthlyCharges", "Churn"]]


11


,customerID,tenure,MonthlyCharges,Churn
488,4472-LVYGI,0,52.55,No
753,3115-CZMZD,0,20.25,No
936,5709-LVOEQ,0,80.85,No
1082,4367-NUYAO,0,25.75,No
1340,1371-DWPAZ,0,56.05,No
3331,7644-OMVMY,0,19.85,No
3826,3213-VVOLG,0,25.35,No
4380,2520-SGTTA,0,20.00,No
5218,2923-ARZLG,0,19.70,No
6670,4075-WKNIU,0,73.35,No


In [4]:
df = df.dropna(subset=["TotalCharges"])
print(df.shape)

(7032, 21)


In [5]:
print(df["Churn"].value_counts())
print(df["Churn"].value_counts(normalize=True).round(3) * 100)

Churn
No     5163
Yes    1869
Name: count, dtype: int64
Churn
No     73.4
Yes    26.6
Name: proportion, dtype: float64


In [6]:
df["churn_flag"] = (df["Churn"] == "Yes").astype(int)

In [7]:
for col in ["Contract", "InternetService", "PaymentMethod", "TechSupport"]:
    print(col)
    print((df.groupby(col)["churn_flag"].mean() * 100).round(1).sort_values(ascending=False))

Contract
Contract
Month-to-month    42.7
One year          11.3
Two year           2.8
Name: churn_flag, dtype: float64
InternetService
InternetService
Fiber optic    41.9
DSL            19.0
No              7.4
Name: churn_flag, dtype: float64
PaymentMethod
PaymentMethod
Electronic check             45.3
Mailed check                 19.2
Bank transfer (automatic)    16.7
Credit card (automatic)      15.3
Name: churn_flag, dtype: float64
TechSupport
TechSupport
No                     41.6
Yes                    15.2
No internet service     7.4
Name: churn_flag, dtype: float64


In [8]:

df["tenure_group"] = pd.cut(df["tenure"], bins=[0, 12, 24, 48, 72],
                            labels=["0-12 months", "13-24 months", "25-48 months", "49-72 months"])

print((df.groupby("tenure_group", observed=True)["churn_flag"].mean() * 100).round(1))

tenure_group
0-12 months     47.7
13-24 months    28.7
25-48 months    20.4
49-72 months     9.5
Name: churn_flag, dtype: float64


In [9]:

print(df.groupby("Churn")["MonthlyCharges"].mean().round(2))

print((df.groupby("SeniorCitizen")["churn_flag"].mean() * 100).round(1))

Churn
No     61.31
Yes    74.44
Name: MonthlyCharges, dtype: float64
SeniorCitizen
0    23.7
1    41.7
Name: churn_flag, dtype: float64


In [10]:
risk = df[(df["Contract"] == "Month-to-month") & (df["tenure"] <= 12)]
print(len(risk), "customers, churn rate:", round(risk["churn_flag"].mean() * 100, 1), "%")

1994 customers, churn rate: 51.4 %


In [11]:
df.to_csv("churn_clean.csv", index=False)
print("Saved")

Saved
